# ArcUI — First Experiment

A 5-minute tour of the Python bridge against a running ArcUI scene.

**Prerequisites**

1. ArcUI project open and in **Play mode**.
2. The ArcUI bridge active in the scene, listening on `http://localhost:17842` (the default; override with `ARCUI_BRIDGE_URL`).
3. The scene loads both your production and research context layers (see the ArcUI SDK setup guide).
4. This notebook launched with the `science` extras installed:
   ```bash
   uv sync --extra science
   uv run --extra science jupyter notebook examples/first_experiment.ipynb
   ```

**What this notebook teaches**

1. The connection — bridge alive, tags discoverable.
2. **Mode-aware contract enforcement** — a scenario that writes across lanes, and how the system behaves differently per runtime mode.
3. **The right pattern** — a scenario writing to a research-owned tag the contract authorizes natively.
4. Reading the chronological record back into pandas.
5. Plotting the captured curve.
6. Live event injection.

**Heads-up:** every cell that calls `bridge.*` writes against your live ArcUI scene. Stopping Play discards all of it.

## 1. Connect

In [ ]:
import asyncio
import pandas as pd
import matplotlib.pyplot as plt
from arcui_mcp.bridge import bridge

try:
    health = await bridge.get_system_health()
    print("Bridge reachable. System health:")
    print(health)
except Exception as e:
    print("Could not reach the ArcUI bridge.")
    print("Is Unity in Play mode with McpBridge active on localhost:17842?")
    print(f"Underlying error: {e}")

## 2. Discover live tags

List every tag currently registered in the runtime. With both context layers loaded you should see your production tags **and** the research tag `Research.ExperimentSetpoint`.

In [ ]:
tags_response = await bridge.list_tags()
raw = tags_response.get("tags", tags_response)
tags_df = pd.DataFrame(raw)
tags_df

## 3. Mode-aware enforcement — a scenario that writes outside its lane

ArcUI's contract enforcement adapts to the runtime mode:

- **Strict mode (production scenes)** — a write from a non-owner writer is *denied* and dropped. An experiment cannot accidentally drive a sensor channel it does not own.
- **Relaxed mode (prototyping / training)** — a write from a non-owner writer is *allowed* and tagged in the console with an override line. This keeps prototyping low-friction while still leaving an audit trail.

The cell below targets a production-owned tag from the `training_scenario` writer. The runtime decides whether the writes land or are denied based on the current mode. **Replace `PRODUCTION_TAG` below with any tag key from your own production context** before running the cell.

In [ ]:
# Replace with any tag key from your production context layer.
PRODUCTION_TAG = "<your.production.tag.key>"

cross_lane_id = "cross_lane_write_demo"

await bridge.create_scenario(
    id=cross_lane_id,
    display_name="Cross-lane write demo (mode-dependent behavior)",
    description=(
        "This scenario intentionally targets a production-owned tag "
        "from the training_scenario writer. Whether the writes land "
        "or are denied depends on the runtime mode of the scene."
    ),
    events=[
        {"offset_seconds": 0, "tag_key": PRODUCTION_TAG, "value_type": "Float", "raw_value": "10"},
        {"offset_seconds": 3, "tag_key": PRODUCTION_TAG, "value_type": "Float", "raw_value": "20"},
    ],
)

await bridge.start_session(procedure="cross_lane_demo")
await bridge.start_scenario(cross_lane_id)
await asyncio.sleep(5)
cross_record = await bridge.evaluate_session()
await bridge.end_session()

cross_df = pd.DataFrame(cross_record.get("events", []))
target_writes = cross_df[
    (cross_df.get('kind') == 'tag_changed') & (cross_df.get('subject') == PRODUCTION_TAG)
] if not cross_df.empty else cross_df

if target_writes.empty:
    print(f"No writes to '{PRODUCTION_TAG}' recorded.")
    print("If you expected the writes to land (relaxed mode), check the scene is configured for prototyping.")
    print("If you expected them to be denied (strict mode), the enforcement path worked as designed.")
else:
    print(f"{len(target_writes)} write(s) landed under the relaxed-enforcement path.")
    print("Check the runtime console for the matching override lines — they are the audit trail of the cross-lane writes.")
    target_writes.head()

## 4. The right pattern — author against a research-owned tag

The research context layer (`Research_Lab_context.json`) declares the writer `training_scenario` and gives it ownership of `Research.ExperimentSetpoint`. Now the same scenario shape, targeting that tag, is **authorized** and will land in the record.

Note how the `description` carries the **hypothesis being tested** — a free-form line that becomes part of the bundle and is readable by collaborators and AI debriefers.

In [ ]:
scenario_id = "research_setpoint_step_01"

await bridge.create_scenario(
    id=scenario_id,
    display_name="Setpoint step response",
    description=(
        "Hypothesis: stepping Research.ExperimentSetpoint by +10 every 3s produces a clean "
        "staircase in the recorded record, with no contract denials. Baseline for any "
        "future research scenario that wants to push the setpoint along a curve."
    ),
    events=[
        {"offset_seconds": 0,  "tag_key": "Research.ExperimentSetpoint", "value_type": "Float", "raw_value": "10"},
        {"offset_seconds": 3,  "tag_key": "Research.ExperimentSetpoint", "value_type": "Float", "raw_value": "20"},
        {"offset_seconds": 6,  "tag_key": "Research.ExperimentSetpoint", "value_type": "Float", "raw_value": "30"},
        {"offset_seconds": 9,  "tag_key": "Research.ExperimentSetpoint", "value_type": "Float", "raw_value": "40"},
        {"offset_seconds": 12, "tag_key": "Research.ExperimentSetpoint", "value_type": "Float", "raw_value": "50"},
    ],
)

await bridge.start_session(procedure="setpoint_step_study")
await bridge.start_scenario(scenario_id)
await asyncio.sleep(14)
record = await bridge.evaluate_session()
ended = await bridge.end_session()
print("Session ended. Bundle written to:", ended.get("bundle_dir"))

## 5. Inspect the chronological record

`evaluate_session` returns events with `ts`, `kind`, `subject`, `value`, `detail`, `writer_id`. Tag writes appear as `kind == "tag_changed"` with `subject == <tag>` and `value == <written value>`. Note the `writer_id` field — it is the writer attribution that the contract authorized (and the column you filter on when separating scenario-injected data from real telemetry).

In [ ]:
events_df = pd.DataFrame(record.get("events", []))
print(f"Event count: {record.get('event_count')}")
if not events_df.empty:
    print("Kinds present:", events_df['kind'].value_counts().to_dict())
events_df.tail(20)

## 6. Plot the captured curve

Extract the writes to `Research.ExperimentSetpoint` and plot the staircase.

In [ ]:
if events_df.empty:
    print("No events captured. Did the scenario run? Check that:")
    print(" - The bridge is reporting healthy.")
    print(" - The scene is configured per the ArcUI SDK setup guide (scenario runner + research context layer).")
else:
    setpoint_writes = events_df[
        (events_df.get('kind') == 'tag_changed') &
        (events_df.get('subject') == 'Research.ExperimentSetpoint')
    ].copy()

    if setpoint_writes.empty:
        print("No Research.ExperimentSetpoint writes in the record.")
        print("Available subjects:", events_df['subject'].dropna().unique())
    else:
        setpoint_writes['ts'] = pd.to_datetime(setpoint_writes['ts'])
        setpoint_writes['value_num'] = pd.to_numeric(setpoint_writes['value'], errors='coerce')

        fig, ax = plt.subplots(figsize=(9, 4))
        ax.plot(setpoint_writes['ts'], setpoint_writes['value_num'], marker='o', linewidth=2, drawstyle='steps-post')
        ax.set_title('Research.ExperimentSetpoint during research_setpoint_step_01')
        ax.set_xlabel('Time')
        ax.set_ylabel('Value')
        ax.grid(True, alpha=0.3)
        plt.tight_layout()
        plt.show()

## 7. Inject a value live

Outside a scripted scenario you can also write single values to authorized tags on demand — the "what if I change X right now" pattern.

In [ ]:
await bridge.start_session(procedure="live_injection_demo")

result = await bridge.inject_event(
    tag_key="Research.ExperimentSetpoint",
    value_type="Float",
    raw_value="75",
)
print("Injection result:", result)

live_record = await bridge.evaluate_session()
await bridge.end_session()

pd.DataFrame(live_record.get("events", [])).tail(5)

## Next steps

### Load and compare runs with `arcui_mcp.analyze`

After `end_session()` returns a `bundle_dir`, load it from disk and compare runs offline — no live connection needed:

```python
from arcui_mcp.analyze import load_bundle, diff_bundles, plot_run

# Load the bundle this notebook just produced (path printed by end_session)
run = load_bundle(ended["bundle_dir"])
print(run.procedure, run.session_id, run.tag_keys)

# Plot the whole run (one line per numeric tag)
fig = plot_run(run)
fig.savefig("run_overview.png")

# Or compare two runs (baseline vs perturbed) for a hypothesis test
baseline = load_bundle("/path/to/baseline_bundle")
perturbed = load_bundle("/path/to/perturbed_bundle")
report = diff_bundles(baseline, perturbed)
print("Tags only in perturbed:", report.tags_only_in_b)
report.tag_summary
```

### Extend the MCP server with your own tools

A worked template lives at `examples/custom_tools/drift_detector.py`. It detects linear drift in a tag's writes across a recorded session — useful for asking "did the setpoint stay flat?" or "is this measurement creeping over time?". Run it standalone:

```bash
uv run --extra science python examples/custom_tools/drift_detector.py \
    "<bundle_dir>" "<tag-key>" 0.01
```

To expose it as an MCP tool, the most reliable option is to **copy the `@mcp.tool()` block from `drift_detector.py` directly into `src/arcui_mcp/server.py`** between the existing tool definitions, then restart the server. The README has the alternative import-based recipe (works only when launching from the repo root).

### Keep going

- **Add your own research tags** in a research context layer of your own. Keep production contexts untouched.
- **For long-running data collection**, prefer reading the bundle directory returned by `end_session` rather than `evaluate_session` — the on-disk bundle survives runtime restarts and is what `load_bundle` is designed for.